# Pseudo-label chipping pipeline (v2)

**Changes from v1:**
- Removed SAR NaN urban check — label coverage used instead
- Collapsed class 1 (flood open) + class 2 (flood urban) → single flood class 1
- Class 3 (dry urban) → class 2 (non-flood)
- Lowered MIN_FLOOD_FRAC to 0.005
- Added per-event label histogram diagnostics
- Added diagnostic cell to inspect individual events before chipping

In [ ]:
# ── Cell 1: Mount Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
RAW_DIR   = '/content/drive/MyDrive/water_detection_pseudolabels'
CHIPS_DIR = '/content/drive/MyDrive/water_detection_chips'
os.makedirs(CHIPS_DIR, exist_ok=True)

raw_files   = sorted(os.listdir(RAW_DIR))
sar_files   = [f for f in raw_files if f.endswith('_SAR.tif')]
label_files = [f for f in raw_files if f.endswith('_LABEL.tif')]
print(f'SAR files:   {len(sar_files)}')
print(f'Label files: {len(label_files)}')
print()
for f in sar_files:
    mb = os.path.getsize(os.path.join(RAW_DIR, f)) / 1e6
    print(f'  {f}  ({mb:.0f} MB)')

In [ ]:
# ── Cell 2: Install rasterio ───────────────────────────────────────────────
!pip install -q rasterio==1.3.10
print('rasterio ready')

In [ ]:
# ── Cell 3: Configuration ──────────────────────────────────────────────────
CHIP_SIZE        = 512

# Quality filters
MIN_FLOOD_FRAC   = 0.005  # ≥ 0.5% flood pixels among valid pixels (relaxed)
MIN_VALID_FRAC   = 0.20   # ≥ 20% non-masked label pixels (relaxed from 30%)
MAX_NODATA_FRAC  = 0.90   # < 90% NaN in SAR — only reject completely empty chips
# NOTE: removed MIN_URBAN_FRAC — urban coverage enforced via label pixels,
# not SAR NaN. The GEE export already restricts labels to the urban zone.

# Label remapping
# GEE exported: 0=masked, 1=flood_open, 2=flood_urban, 3=dry_urban
# We remap to:  0=masked, 1=flood (any), 2=non_flood (dry urban)
# Rationale: WorldCover built-up class too coarse to reliably split
# flood into open/urban at pixel level. Single flood class is cleaner.
REMAP = {0: 0, 1: 1, 2: 1, 3: 2}

EVENTS = [
    'UK_Yorkshire_2015',
    'Zhengzhou_China_2021',
    'Pakistan_Sindh_2022',
    'Cyclone_Gabrielle_NZ_2023',
    'Jeddah_Saudi_2022',
    'Bucharest_Romania_2024',
    'Seville_Spain_2025',
    'Cologne_Rhine_2021',
    'Western_Europe_2021',
    'Perth_Australia_2021',
    'Libya_Derna_2023',
    'Seoul_South_Korea_2022',
    'Typhoon_Vamco_Manila_2020',
    'Hurricane_Florence_2018',
]

print(f'Events: {len(EVENTS)}')
print(f'Label remap: {REMAP}')
print(f'MIN_FLOOD_FRAC={MIN_FLOOD_FRAC}, MIN_VALID_FRAC={MIN_VALID_FRAC}')

In [ ]:
# ── Cell 4: Diagnostic — inspect label histogram for each event ────────────
# Run this before chipping to understand what's in each export.
# Shows raw GEE label counts (before remapping) so we can see:
#   - How many flood pixels exist (class 1+2)
#   - Whether the urban zone masking is working (non-zero class 3)
#   - Whether any events have zero flood signal (need re-export)

import numpy as np
import rasterio
from pathlib import Path
from collections import Counter

print(f'{"Event":<40} {"0_masked":>12} {"1_flood_open":>14} '
      f'{"2_flood_urban":>15} {"3_dry":>10} {"flood%":>8}')
print('-' * 105)

problem_events = []

for event in EVENTS:
    lbl_path = Path(RAW_DIR) / f'{event}_LABEL.tif'
    if not lbl_path.exists():
        print(f'{event:<40} MISSING')
        problem_events.append(event)
        continue

    with rasterio.open(lbl_path) as src:
        # Sample at reduced resolution for speed (every 4th pixel)
        data = src.read(1, out_shape=(
            max(1, src.height // 4),
            max(1, src.width  // 4)
        )).ravel()

    counts = Counter(data.tolist())
    total  = len(data)
    c0 = counts.get(0, 0)
    c1 = counts.get(1, 0)
    c2 = counts.get(2, 0)
    c3 = counts.get(3, 0)
    flood_pct = 100 * (c1 + c2) / max(total, 1)

    flag = '  ← NO FLOOD' if (c1 + c2) == 0 else ''
    print(f'{event:<40} {c0:>12,} {c1:>14,} {c2:>15,} '
          f'{c3:>10,} {flood_pct:>7.2f}%{flag}')

    if (c1 + c2) == 0:
        problem_events.append(event)

print()
if problem_events:
    print(f'Events with no flood signal: {problem_events}')
    print('These need re-export from GEE with a lower NDWI threshold.')
else:
    print('All events have flood signal. Proceed to Cell 5.')

In [ ]:
# ── Cell 5: Dry run ────────────────────────────────────────────────────────
from rasterio.windows import Window

# Vectorised label remap using lookup table
REMAP_TABLE = np.zeros(256, dtype=np.uint8)
for k, v in REMAP.items():
    REMAP_TABLE[k] = v

def chip_event(event, raw_dir, chips_dir, cap=None, dry_run=False):
    sar_path   = Path(raw_dir)  / f'{event}_SAR.tif'
    label_path = Path(raw_dir)  / f'{event}_LABEL.tif'
    out_dir    = Path(chips_dir)

    if not sar_path.exists():
        return {'event': event, 'skipped': True, 'reason': 'SAR not found'}
    if not label_path.exists():
        return {'event': event, 'skipped': True, 'reason': 'LABEL not found'}

    stats = dict(
        event=event, skipped=False,
        total=0, accepted=0, rej_nodata=0, rej_valid=0, rej_flood=0, rej_cap=0,
        flood_px=0, nonfloood_px=0,
    )

    with rasterio.open(sar_path) as sar_src, \
         rasterio.open(label_path) as lbl_src:

        h, w   = sar_src.height, sar_src.width
        n_rows = h // CHIP_SIZE
        n_cols = w // CHIP_SIZE

        for row in range(n_rows):
            for col in range(n_cols):
                stats['total'] += 1

                if cap and stats['accepted'] >= cap:
                    stats['rej_cap'] += 1
                    continue

                win = Window(
                    col_off=col * CHIP_SIZE, row_off=row * CHIP_SIZE,
                    width=CHIP_SIZE, height=CHIP_SIZE
                )
                sar_chip = sar_src.read(window=win).astype(np.float32)
                lbl_raw  = lbl_src.read(1, window=win).astype(np.uint8)

                # Remap labels: 0=masked, 1=flood, 2=non_flood
                lbl_chip = REMAP_TABLE[lbl_raw]

                # Filter 1: SAR completely empty (all NaN)
                nodata_frac = float(np.isnan(sar_chip).mean())
                if nodata_frac > MAX_NODATA_FRAC:
                    stats['rej_nodata'] += 1
                    continue

                # Filter 2: not enough labeled pixels
                total_px = CHIP_SIZE * CHIP_SIZE
                valid_px = int((lbl_chip != 0).sum())
                if valid_px / total_px < MIN_VALID_FRAC:
                    stats['rej_valid'] += 1
                    continue

                # Filter 3: not enough flood pixels
                flood_px = int((lbl_chip == 1).sum())
                if flood_px / max(valid_px, 1) < MIN_FLOOD_FRAC:
                    stats['rej_flood'] += 1
                    continue

                stats['accepted']     += 1
                stats['flood_px']     += flood_px
                stats['nonfloood_px'] += int((lbl_chip == 2).sum())

                if dry_run:
                    continue

                stem      = f'{event}_{row:04d}_{col:04d}'
                transform = sar_src.window_transform(win)

                with rasterio.open(
                    out_dir / f'{stem}_SAR.tif', 'w',
                    driver='GTiff', height=CHIP_SIZE, width=CHIP_SIZE,
                    count=2, dtype=np.float32,
                    crs=sar_src.crs, transform=transform
                ) as dst:
                    dst.write(sar_chip)

                with rasterio.open(
                    out_dir / f'{stem}_LABEL.tif', 'w',
                    driver='GTiff', height=CHIP_SIZE, width=CHIP_SIZE,
                    count=1, dtype=np.uint8,
                    crs=lbl_src.crs, transform=transform
                ) as dst:
                    dst.write(lbl_chip[np.newaxis])

    return stats


print('DRY RUN — no files written\n')
all_stats = []

for event in EVENTS:
    s = chip_event(event, RAW_DIR, CHIPS_DIR, cap=None, dry_run=True)
    if s.get('skipped'):
        print(f'{event}: SKIPPED — {s["reason"]}')
    else:
        acc = s['accepted']
        tot = s['total']
        print(f'{event:<40} {acc:4d}/{tot} chips  '
              f'flood={s["flood_px"]:,}px  '
              f'dry={s["nonfloood_px"]:,}px  '
              f'rej(nodata={s["rej_nodata"]} valid={s["rej_valid"]} flood={s["rej_flood"]})')
    all_stats.append(s)

total_acc   = sum(s.get('accepted', 0) for s in all_stats)
total_chips = sum(s.get('total', 0)    for s in all_stats)
total_flood = sum(s.get('flood_px', 0) for s in all_stats)
total_dry   = sum(s.get('nonfloood_px', 0) for s in all_stats)

sar_mb   = CHIP_SIZE * CHIP_SIZE * 2 * 4 / 1e6
lbl_mb   = CHIP_SIZE * CHIP_SIZE * 1 * 1 / 1e6
est_gb   = total_acc * (sar_mb + lbl_mb) / 1e3

print(f'\n{"="*56}')
print(f'TOTAL: {total_acc}/{total_chips} chips ({100*total_acc/max(total_chips,1):.1f}%)')
print(f'Flood pixels : {total_flood:,}')
print(f'Dry pixels   : {total_dry:,}')
print(f'Estimated size: ~{est_gb:.2f} GB (×1.2 with GeoTIFF overhead)')

In [ ]:
# ── Cell 6: Write chips ────────────────────────────────────────────────────
# Only run after reviewing dry run output.
# Cap per event to balance dataset — adjust as needed.

import time
MAX_CHIPS_PER_EVENT = 500  # increase if total chips < 1000

print(f'Writing chips (cap={MAX_CHIPS_PER_EVENT} per event)...\n')
t0 = time.time()
all_stats = []

for event in EVENTS:
    s = chip_event(event, RAW_DIR, CHIPS_DIR,
                   cap=MAX_CHIPS_PER_EVENT, dry_run=False)
    acc = s.get('accepted', 0)
    all_stats.append(s)
    status = 'SKIPPED' if s.get('skipped') else f'{acc} chips'
    print(f'  {event:<40} {status}')

elapsed     = time.time() - t0
total_acc   = sum(s.get('accepted', 0) for s in all_stats)
total_flood = sum(s.get('flood_px', 0) for s in all_stats)

print(f'\nDone in {elapsed/60:.1f} min')
print(f'Total chips written: {total_acc}')
print(f'Total flood pixels:  {total_flood:,}')

In [ ]:
# ── Cell 7: Verify and summarise ──────────────────────────────────────────
from collections import Counter

chip_files = sorted(Path(CHIPS_DIR).glob('*_SAR.tif'))
print(f'Total SAR chips: {len(chip_files)}')

counts = Counter()
for f in chip_files:
    # filename: {event_parts}_{row}_{col}_SAR.tif
    # last 3 underscore-separated parts before _SAR are row, col, SAR
    parts = f.stem.split('_')[:-3]  # strip row, col, SAR
    counts['_'.join(parts)] += 1

print('\nPer-event:')
for e, c in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {e:<42} {c:5d}')

total_gb = sum(os.path.getsize(f) for f in Path(CHIPS_DIR).glob('*.tif')) / 1e9
print(f'\nTotal size: {total_gb:.2f} GB')
print('\nDownload water_detection_chips/ from Drive to:')
print('  ~/Projects/water-detection/data/pseudo_labels/chips/')